# 04 · Scale & Thresholds Lab

**Purpose.** Исследовать нестабильность шкалы: production vs backtest, дрейф при разных training cutoffs, идея ретроактивного MinMax и перцентильных порогов.

**What to look for.**
- насколько backtest-шкала отличается от production (Dec2014!)
- дрейф уровня при обучении на разных cutoff
- какие пороги нужны, чтобы поймать каждый эпизод
- перцентильные пороги vs фиксированные 30/60

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
from lab import utils as u
print('project root:', _root.name)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
CUTOFFS = ['2018-12-31','2020-12-31','2022-12-31','2024-12-31']
EPISODES = u.STRESS_EPISODES
THRESHOLDS = u.DEFAULT_THRESHOLDS
PERCENTILES = [80, 90, 95, 99]

In [ ]:
d = u.load_final_dataset()
av = u.available_whitelist(d)
base = u.fit_lsi_like_model(d, av)
baseline = base['lsi']
prod = pd.DataFrame({'date': base['date'], 'lsi': baseline})

### Production vs backtest scores в эпизодах

In [ ]:
bt = u.load_backtest_scores()
prod_ep = u.compute_episode_summary(prod['date'], prod['lsi'], EPISODES).rename(columns={'max':'prod_max','mean':'prod_mean'})
bt_rows = []
name_map = {'Dec2014':'Декабрь 2014','Feb-Mar2022':'Февраль-март 2022','Aug2023':'Август 2023'}
for k,ru in name_map.items():
    s = bt[bt['episode']==ru]['lsi_global_backtest']
    bt_rows.append({'episode':k,'bt_max':round(float(s.max()),2) if len(s) else None,'bt_mean':round(float(s.mean()),2) if len(s) else None})
prod_ep[['episode','prod_max','prod_mean']].merge(pd.DataFrame(bt_rows), on='episode')

### Дрейф шкалы при разных training cutoffs
Переобучаем Global только на данных до cutoff и сравниваем уровень на общем перекрытии.

In [ ]:
from backend.src.services.lsi_training_service import fit_lsi_artifact
drift = []
for c in CUTOFFS:
    sub = d[d['date'] <= c].reset_index(drop=True)
    _, sc = fit_lsi_artifact(sub, kind='global')
    lc = sc['lsi_global'].to_numpy()
    canon = baseline[:len(sub)]
    cmp = u.compare_scores(lc, canon)
    drift.append({'cutoff':c,'n':len(sub),'spearman_vs_full':round(cmp['spearman'],4),
                  'mean_level_diff':round(cmp['mean_diff'],2),'max_abs_diff':round(cmp['max_abs_diff'],2)})
pd.DataFrame(drift)

### Идея: ретроактивный MinMax (одна шкала на полную историю)
Здесь baseline уже full-history, поэтому показываем, как меняется уровень, если нормировать smoothed-score по перцентилям вместо min/max.

In [ ]:
sm = base['smoothed']
# robust scaling: 1й и 99й перцентиль вместо min/max
lo, hi = np.percentile(sm, 1), np.percentile(sm, 99)
robust = np.clip((sm - lo)/(hi - lo)*100, 0, 100)
fig, ax = u.plot_lsi_series(base['date'], {'MinMax (current)': baseline, 'robust 1-99 pctl': robust},
    episodes=u.STRESS_EPISODES, thresholds=THRESHOLDS, title='MinMax vs robust scaling'); plt.show()
print('Spearman(minmax, robust):', round(u.spearman(baseline, robust),4))

### Перцентильные пороги vs фиксированные 30/60

In [ ]:
pct_table = pd.DataFrame({'percentile': PERCENTILES,
    'lsi_value': [round(float(np.percentile(baseline, p)),2) for p in PERCENTILES]})
pct_table

### Какой порог нужен, чтобы поймать каждый эпизод (как RED)

In [ ]:
need = []
for k,(a,b) in EPISODES.items():
    seg = prod[(prod['date']>=a)&(prod['date']<=b)]['lsi']
    if len(seg):
        need.append({'episode':k,'episode_max':round(float(seg.max()),2),
                     'pctile_of_max': round(float((baseline < seg.max()).mean()*100),1)})
pd.DataFrame(need)

## Notes / Open questions

- Dec2014: backtest даёт RED, production — нет. Это не баг модели, а разные MinMax-fit.
- Дрейф уровня по cutoff подтверждает: абсолютная шкала не сопоставима во времени.
- Перцентильный якорь (например, RED = p95+) устойчивее к переобучению, чем фиксированный 60.